# Week 1 — Research Question

**Author:** Zain-ul-Abdeen  
**Date:** 2026-07-09  
**Repo:** [Zain-ul-abdeen-773/flyrank-ml-internship](https://github.com/Zain-ul-abdeen-773/flyrank-ml-internship)

---
## 1. My Lane (and Why)

**Selected lane: Lane 4 — CTR / Engagement Opportunity Scoring**

The question from the lane guide:

> *Which visible pages under-capture clicks or engagement and deserve metadata, content, or monitoring review?*

### Why this lane?

After running the starter notebooks and inspecting the data, three observations pulled me toward CTR opportunity scoring:

1. **There is a huge, concrete click gap.** Over 12,000 pages in the starter slice sit at position ≤ 20 (i.e. on page 1 or in "striking distance") yet have a CTR below 0.5%. Together those pages account for over 91 million impressions in the 90-day window — impressions that result in almost no clicks. If even a fraction of those pages could be improved, the aggregate click gain is large.

2. **Position alone doesn't explain CTR.** The starter notebook showed a clear *average* CTR drop as position worsens, but inside every position tier there is enormous variance. Nearly half (46.8%) of visible pages fall below their own position-tier's median CTR. That variance is the signal a model or scored ranking can try to explain and prioritize.

3. **The action is clear and low-risk.** When a page is ranked well but not getting clicked, the most likely levers are the title tag, meta description, snippet structure, or on-page engagement quality. Those are cheap, reversible actions a content team can take without rewriting the entire article. The worst case of a wrong recommendation is that a reviewer spends a few minutes inspecting a page that turns out to be fine — not a damaging action.

I also considered Lane 2 (Refresh Scoring) because the starter pipeline already demonstrates it end-to-end. But the starter's label (`trend_direction == "down"`) is a same-window proxy, not a future outcome. Lane 4 avoids that weakness because the target is a *current observable gap* (under-performing CTR relative to position) rather than a trend prediction.

---
## 2. The Question: Decision, Action, and Cost of a Wrong Call

### The search question

> **Among pages that are visible in search (enough impressions, a known position), which ones are under-capturing clicks relative to what their position should deliver, and how should they be prioritized for review?**

### Unit of analysis

**One content page** (one row in the starter dataset, keyed by `content_id`), filtered to pages with at least 100 impressions in the 90-day window and a non-zero `avg_position` — the population where CTR is actually measurable.

### Output

A **ranked list of CTR-opportunity pages**, each scored by how far its actual CTR falls below what the position tier (and other observable features) would predict. Each entry carries:
- an **opportunity score** (higher = bigger gap, more worth reviewing),
- a **reason code** (e.g. "high impressions, low CTR", "strong position, weak snippet performance"),
- a **suggested action** (e.g. rewrite title/meta, improve intent match, monitor).

### The decision it improves

A content or SEO team has limited time. They cannot review every page. Today, without a model, they might sort by impressions or scan manually. This scored ranking tells them: *review these pages first — they have the most unrealized click potential given where they already rank.*

### The action someone takes

For a high-opportunity page, the reviewer can:
- **Rewrite the title tag or meta description** to better match the searcher's intent.
- **Improve snippet-eligible structured content** (headings, FAQ blocks, lists).
- **Check intent mismatch** — is the page targeting an informational query but written as transactional, or vice versa?
- **Improve on-page engagement** if scroll/engagement data also looks weak.
- **Monitor** if the gap is small or the volume is low.

### Cost of a wrong recommendation

| Error type | What happens | Severity |
|---|---|---|
| **False positive** (ranked high, but CTR is actually normal for this page) | Reviewer spends 5–10 minutes checking a page that didn't need work. Low cost, especially if reason codes let them dismiss it quickly. | Low |
| **False negative** (missed a real opportunity) | A page that *could* have gained clicks stays unreviewed. The cost scales with impression volume — a missed page with 50,000 impressions is worse than one with 200. | Medium |
| **Misattribution** (CTR gap is due to SERP features or intent shift, not the page itself) | The action taken (rewriting the title) may not help. But the action itself is low-cost and reversible. | Low–Medium |

The overall risk is **low**: the worst outcomes are wasted reviewer time or an ineffective edit, not damage to the content.

### Why data / ML can help at all

A simple rule like "find pages with CTR below X" ignores position: a CTR of 0.3% is poor for a top-3 page but excellent for a page at position 40. The right benchmark is *position-adjusted expected CTR*, and even that varies by content type, intent, impression volume, and engagement context. A model can learn that multi-dimensional expected-CTR surface far more accurately than any single threshold, producing a ranked queue where the biggest gaps rise to the top.

---
## 3. Quick Look at the Data (Supporting Numbers)

All numbers below are computed live from the starter dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 rows × 44 columns, 32 clients).

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Zain-ul-abdeen-773/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    import subprocess
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir(os.path.join("..", ".."))  # move from work/notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found.")

Working dir: D:\Study\New folder (2)
Starter data found.


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns,", df["client_id"].nunique(), "clients")

30000 rows, 44 columns, 32 clients


### Evidence 1 — The CTR cliff is real, but the *variance within tiers* is enormous

Mean CTR drops predictably from top-3 → deep, confirming that position matters. But the gap between the *median* and the *mean* inside each tier — and the wide range — shows that position explains only part of CTR. That leftover variance is the opportunity.

In [3]:
# Filter to visible pages: at least 100 impressions in the 90-day window
visible = df[df["impressions_90d"] >= 100].copy()
print("Visible pages (impressions >= 100):", len(visible), "out of", len(df))

# CTR stats by position tier
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
ctr_stats = visible.groupby("position_tier")["ctr"].agg(["mean", "median", "std", "count"])
ctr_stats = ctr_stats.reindex(tier_order)
print("\nCTR by position tier (visible pages):")
print(ctr_stats.round(3).to_string())

Visible pages (impressions >= 100): 22006 out of 30000

CTR by position tier (visible pages):
                mean  median    std  count
position_tier                             
top_3          0.334    0.19  0.487    533
page_1         0.355    0.23  0.502   8633
striking       0.256    0.15  0.347   5903
page_3_5       0.142    0.06  0.229   6058
deep           0.055    0.00  0.170    879


### Evidence 2 — 12,000+ pages sit at a good position but barely get clicked

Pages with `avg_position <= 20` (page 1 or striking distance) and `CTR < 0.5%` represent a pool of over **91 million impressions** that convert to almost no clicks. That is the core opportunity pool this lane would score and rank.

In [4]:
# Pages with good position but very low CTR
has_pos = visible[(visible["avg_position"] > 0) & (visible["avg_position"] <= 20)]
low_ctr = has_pos[has_pos["ctr"] < 0.5]
print("Pages at position <= 20 with CTR < 0.5%:", len(low_ctr))
total_imp = low_ctr["impressions_90d"].sum()
print("Their total 90-day impressions:", f"{total_imp:,.0f}")
print("Median impressions per page:", f"{low_ctr['impressions_90d'].median():,.0f}")
print()

# For comparison: what share of ALL visible pages is this?
share = len(low_ctr) / len(visible) * 100
print(f"That is {share:.1f}% of all visible pages — a large, actionable pool.")

Pages at position <= 20 with CTR < 0.5%: 12124
Their total 90-day impressions: 91,616,766
Median impressions per page: 2,018

That is 55.1% of all visible pages — a large, actionable pool.


### Evidence 3 — Nearly half of visible pages fall below their position-tier's median CTR

Even a simple position-adjusted benchmark (median CTR per tier) reveals that ~47% of visible pages under-perform. A model that uses additional features (content type, intent, word count, engagement signals) could produce a more accurate expected-CTR estimate and a more useful ranking.

In [5]:
# Median CTR per position tier = simple position-adjusted expected CTR
expected = visible.groupby("position_tier")["ctr"].median()
print("Median CTR by position tier (simple expected CTR):")
print(expected.reindex(tier_order).round(3).to_string())

# Merge expected CTR and flag under-performers
merged = visible.merge(
    expected.reset_index().rename(columns={"ctr": "expected_ctr"}),
    on="position_tier"
)
below = merged[merged["ctr"] < merged["expected_ctr"]]
pct = len(below) / len(visible) * 100
print(f"\nPages below their tier's median CTR: {len(below)} ({pct:.1f}% of visible pages)")
print(f"Impressions represented: {below['impressions_90d'].sum():,.0f}")
print()
print("A model can do better than tier medians by using content type, intent,")
print("word count, freshness, engagement, and other observable features.")

Median CTR by position tier (simple expected CTR):


position_tier
top_3       0.19
page_1      0.23
striking    0.15
page_3_5    0.06
deep        0.00

Pages below their tier's median CTR: 10307 (46.8% of visible pages)
Impressions represented: 58,170,533

A model can do better than tier medians by using content type, intent,
word count, freshness, engagement, and other observable features.


### Bonus: content type matters within the same position tier

Among page-1 / top-3 pages, `comparison article` pages have a dramatically different CTR profile than `keyword article` or `feedly article` pages. A flat CTR threshold would mis-rank these.

In [6]:
# CTR by content type — within page_1 + top_3 only
good_pos = visible[visible["position_tier"].isin(["page_1", "top_3"])]
ctr_by_type = good_pos.groupby("content_type")["ctr"].agg(["mean", "median", "count"])
print("CTR by content type (page-1 / top-3 pages only):")
print(ctr_by_type.round(3).to_string())
print()
print("Comparison articles at top positions have median CTR of 0.00 — very different")
print("from keyword articles (0.23). A position-only rule would miss this.")

CTR by content type (page-1 / top-3 pages only):


                     mean  median  count
content_type                            
comparison article  0.141    0.00    227
feedly article      0.949    0.34    226
keyword article     0.344    0.23   8713

Comparison articles at top positions have median CTR of 0.00 — very different
from keyword articles (0.23). A position-only rule would miss this.


---
## 4. Careful Words: What I Can and Can't Claim

### What I can say (observed / directional)

- In this 30,000-row starter slice, nearly half of visible pages have a CTR below their position-tier median. This is an **observed pattern**, not a law of nature.
- Pages with strong positions but very low CTR exist in large numbers (12,000+). This is a **measurable, actionable pool** for review.
- Content type appears to moderate the position → CTR relationship: comparison articles at top positions behave differently from keyword articles. This is **directional** evidence that a multi-feature model could outperform a position-only rule.

### What I cannot claim

- **I cannot claim that rewriting a title or meta description will *cause* CTR to improve.** That requires an experiment (A/B test or time-series comparison with a proper counterfactual). This project is decision-support: it ranks pages *worth reviewing*, not pages *guaranteed to recover*.
- **I cannot claim these patterns generalize beyond this slice.** The starter dataset is 30,000 rows from 32 clients. The full warehouse release (~79M rows, 70 clients) may show different distributions, and any model trained here must be re-validated there.
- **I cannot claim this proves anything about Google's ranking algorithm.** CTR is an observable metric; why it varies within a position tier could involve SERP features, brand recognition, query intent, snippet formatting, or factors outside the data.
- **I cannot treat CTR as a quality metric in isolation.** A page might have low CTR because the SERP snippet already answers the question (a zero-click query), not because the title is bad. Reason codes and reviewer judgment are essential.

### Language discipline

Throughout this project I will describe findings as *observed*, *measured*, *directional*, or *associated with* — never as *proven*, *caused by*, or *will improve*.

---
## 5. Self-Check

| Check | Answer |
|---|---|
| Did I pick a lane (or declare freestyle)? | ✅ Lane 4: CTR / Engagement Opportunity Scoring |
| Did I name the decision? | ✅ Which pages to review first for click-capture improvement, given limited team capacity |
| Did I name the action? | ✅ Rewrite title/meta, improve snippet structure, check intent match, improve engagement, or monitor |
| Did I describe the cost of a wrong call? | ✅ False positive = wasted reviewer time (low). False negative = missed opportunity scaled by impression volume (medium). Misattribution = ineffective but reversible edit (low–medium). |
| Did I show at least 2 real numbers from the starter data? | ✅ Three evidence cells: (1) CTR by position tier with variance, (2) 12,124 pages at position ≤ 20 with CTR < 0.5% totaling 91.6M impressions, (3) 46.8% of visible pages below their tier median CTR |
| Is this "not just train a model"? | ✅ The goal is a ranked review queue with reason codes. The model is a tool to produce a better ranking than a flat rule. Human review is always the last step. |
| Am I using careful language? | ✅ Observed, measured, directional — no causal claims, no "proof" |
| Can I confirm or change my lane until Week 4? | ✅ Yes — this is provisional |